# 01 Data Inventory And Quality

Purpose: verify the CMS raw-data foundation before EDA or modeling.

This notebook checks raw file counts, expected monthly coverage, ZIP integrity, file contents inside each ZIP, and a small readable sample from the MA SCP and CPSC datasets.

## Why This Notebook Exists

For an ML project, the first deliverable is not the model. It is proof that the source data is real, complete enough for the planned task, and readable. If this notebook passes, the rest of the pipeline can safely move into ingestion, EDA, forecasting, and PA-risk modeling.

In [1]:
from __future__ import annotations

import re
import zipfile
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
REPORT_TABLE_DIR = ROOT / "reports" / "tables"
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

MA_SCP_DIR = RAW_DIR / "ma_scp"
CPSC_DIR = RAW_DIR / "cpsc"
MANUAL_DIR = RAW_DIR / "manual_exports"

EXPECTED_MONTHS = pd.period_range("2024-01", "2026-05", freq="M").astype(str).tolist()

print(f"Project root: {ROOT}")
print(f"Expected monthly window: {EXPECTED_MONTHS[0]} to {EXPECTED_MONTHS[-1]} ({len(EXPECTED_MONTHS)} months)")

Project root: D:\Project 1\rcm-cms-mvp
Expected monthly window: 2024-01 to 2026-05 (29 months)


## Raw File Inventory

This section summarizes every raw file currently installed. Raw files are intentionally ignored by git because they are large and reproducible from the download scripts.

In [2]:
def file_inventory(base_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(base_dir.rglob("*")):
        if not path.is_file():
            continue
        rel = path.relative_to(ROOT).as_posix()
        rows.append(
            {
                "relative_path": rel,
                "folder": path.parent.relative_to(RAW_DIR).as_posix() if RAW_DIR in path.parents else "",
                "suffix": path.suffix.lower(),
                "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
            }
        )
    return pd.DataFrame(rows)


inventory = file_inventory(RAW_DIR)
inventory_summary = (
    inventory.groupby(["folder", "suffix"], as_index=False)
    .agg(file_count=("relative_path", "count"), total_mb=("size_mb", "sum"))
    .sort_values(["folder", "suffix"])
)

inventory.to_csv(REPORT_TABLE_DIR / "raw_file_inventory.csv", index=False)
inventory_summary.to_csv(REPORT_TABLE_DIR / "raw_file_inventory_summary.csv", index=False)

display(inventory_summary)
print(f"Raw files: {len(inventory):,}")
print(f"Raw size: {inventory['size_mb'].sum() / 1024:.2f} GB")

,folder,suffix,file_count,total_mb
0,benefits,.zip,1,70.070
1,cpsc,.zip,29,1060.360
2,ma_penetration,.zip,1,0.081
3,ma_scp,.zip,29,4.185
4,manual_exports,.csv,1,0.001
5,manual_exports,.pdf,2,0.288
6,plan_directory,.zip,1,0.231
7,reference_docs,.pdf,3,4.387


Raw files: 67
Raw size: 1.11 GB


## ZIP Integrity And Contents

Each ZIP is opened with Python's `zipfile` module. This catches corrupt or partial downloads before they create hard-to-debug ingestion errors.

In [3]:
def inspect_zip(path: Path) -> dict:
    row = {
        "relative_path": path.relative_to(ROOT).as_posix(),
        "file_name": path.name,
        "folder": path.parent.relative_to(RAW_DIR).as_posix(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
        "is_valid_zip": False,
        "entry_count": 0,
        "csv_count": 0,
        "xlsx_count": 0,
        "txt_count": 0,
        "largest_entry": None,
        "largest_entry_mb": None,
        "error": None,
    }
    try:
        with zipfile.ZipFile(path) as zf:
            entries = [e for e in zf.infolist() if not e.is_dir()]
            row["is_valid_zip"] = True
            row["entry_count"] = len(entries)
            row["csv_count"] = sum(e.filename.lower().endswith(".csv") for e in entries)
            row["xlsx_count"] = sum(e.filename.lower().endswith((".xlsx", ".xls")) for e in entries)
            row["txt_count"] = sum(e.filename.lower().endswith(".txt") for e in entries)
            if entries:
                largest = max(entries, key=lambda e: e.file_size)
                row["largest_entry"] = largest.filename
                row["largest_entry_mb"] = round(largest.file_size / (1024 * 1024), 3)
    except Exception as exc:
        row["error"] = str(exc)
    return row


zip_paths = sorted(RAW_DIR.rglob("*.zip"))
zip_quality = pd.DataFrame([inspect_zip(path) for path in zip_paths])
zip_quality.to_csv(REPORT_TABLE_DIR / "zip_integrity_summary.csv", index=False)

display(zip_quality.groupby(["folder", "is_valid_zip"], as_index=False).agg(zip_count=("file_name", "count"), total_mb=("size_mb", "sum")))
display(zip_quality.head())

bad_zips = zip_quality.loc[~zip_quality["is_valid_zip"]]
assert bad_zips.empty, f"Found bad ZIP files: {bad_zips['relative_path'].tolist()}"
print(f"ZIP files checked: {len(zip_quality):,}; bad ZIPs: {len(bad_zips):,}")

,folder,is_valid_zip,zip_count,total_mb
0,benefits,True,1,70.070
1,cpsc,True,29,1060.360
2,ma_penetration,True,1,0.081
3,ma_scp,True,29,4.185
4,plan_directory,True,1,0.231


,relative_path,file_name,folder,size_mb,is_valid_zip,entry_count,csv_count,xlsx_count,txt_count,largest_entry,largest_entry_mb,error
0,data/raw/benefits/pbp-benefits-2026-json.zip,pbp-benefits-2026-json.zip,benefits,70.070,True,8083,0,0,0,H5522-815-000-2026.json,0.430,None
1,data/raw/cpsc/monthly-enrollment-cpsc-2024-01.zip,monthly-enrollment-cpsc-2024-01.zip,cpsc,37.725,True,3,2,0,1,CPSC_Enrollment_2024_01/CPSC_Enrollment_Info_2024_01.csv,178.099,None
2,data/raw/cpsc/monthly-enrollment-cpsc-2024-02.zip,monthly-enrollment-cpsc-2024-02.zip,cpsc,37.579,True,3,2,0,1,CPSC_Enrollment_2024_02/CPSC_Enrollment_Info_2024_02.csv,177.369,None
3,data/raw/cpsc/monthly-enrollment-cpsc-2024-03.zip,monthly-enrollment-cpsc-2024-03.zip,cpsc,37.555,True,3,2,0,1,CPSC_Enrollment_2024_03/CPSC_Enrollment_Info_2024_03.csv,177.280,None
4,data/raw/cpsc/monthly-enrollment-cpsc-2024-04.zip,monthly-enrollment-cpsc-2024-04.zip,cpsc,37.548,True,3,2,0,1,CPSC_Enrollment_2024_04/CPSC_Enrollment_Info_2024_04.csv,177.240,None


ZIP files checked: 61; bad ZIPs: 0


## Monthly Coverage Checks

The MVP currently expects 29 months of MA SCP and CPSC data: January 2024 through May 2026.

In [4]:
def month_from_name(path: Path) -> str | None:
    match = re.search(r"(20\d{2}-\d{2})", path.name)
    return match.group(1) if match else None


def monthly_coverage(folder: Path, label: str) -> pd.DataFrame:
    found = {month_from_name(path): path.name for path in folder.glob("*.zip") if month_from_name(path)}
    rows = []
    for month in EXPECTED_MONTHS:
        rows.append({"dataset": label, "month": month, "present": month in found, "file_name": found.get(month)})
    return pd.DataFrame(rows)


coverage = pd.concat(
    [monthly_coverage(MA_SCP_DIR, "ma_scp"), monthly_coverage(CPSC_DIR, "cpsc")],
    ignore_index=True,
)
coverage.to_csv(REPORT_TABLE_DIR / "monthly_data_coverage.csv", index=False)

coverage_summary = coverage.groupby("dataset", as_index=False).agg(
    expected_months=("month", "count"),
    present_months=("present", "sum"),
)
coverage_summary["missing_months"] = coverage_summary["expected_months"] - coverage_summary["present_months"]

display(coverage_summary)
display(coverage.loc[~coverage["present"]])

assert coverage["present"].all(), "One or more expected monthly ZIP files are missing."
print("Monthly coverage is complete for MA SCP and CPSC.")

,dataset,expected_months,present_months,missing_months
0,cpsc,29,29,0
1,ma_scp,29,29,0


,dataset,month,present,file_name


Monthly coverage is complete for MA SCP and CPSC.


## Inspect ZIP Entry Patterns

This confirms what each monthly ZIP contains and helps design the ingestion scripts.

In [5]:
entry_examples = zip_quality.loc[
    zip_quality["folder"].isin(["ma_scp", "cpsc"]),
    ["folder", "file_name", "entry_count", "csv_count", "txt_count", "largest_entry", "largest_entry_mb"],
].copy()

display(entry_examples.groupby("folder").head(5))

,folder,file_name,entry_count,csv_count,txt_count,largest_entry,largest_entry_mb
1,cpsc,monthly-enrollment-cpsc-2024-01.zip,3,2,1,CPSC_Enrollment_2024_01/CPSC_Enrollment_Info_2024_01.csv,178.099
2,cpsc,monthly-enrollment-cpsc-2024-02.zip,3,2,1,CPSC_Enrollment_2024_02/CPSC_Enrollment_Info_2024_02.csv,177.369
3,cpsc,monthly-enrollment-cpsc-2024-03.zip,3,2,1,CPSC_Enrollment_2024_03/CPSC_Enrollment_Info_2024_03.csv,177.280
4,cpsc,monthly-enrollment-cpsc-2024-04.zip,3,2,1,CPSC_Enrollment_2024_04/CPSC_Enrollment_Info_2024_04.csv,177.240
5,cpsc,monthly-enrollment-cpsc-2024-05.zip,3,2,1,CPSC_Enrollment_2024_05/CPSC_Enrollment_Info_2024_05.csv,177.004
31,ma_scp,ma-scp-2024-01.zip,2,1,1,SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv,1.016
32,ma_scp,ma-scp-2024-02.zip,2,1,1,SCP_Enrollment_MA_2024_02/SCP_Enrollment_MA_2024_02.csv,1.013
33,ma_scp,ma-scp-2024-03.zip,2,1,1,SCP_Enrollment_MA_2024_03/SCP_Enrollment_MA_2024_03.csv,1.010
34,ma_scp,ma-scp-2024-04.zip,1,1,0,SCP_Enrollment_MA_2024_04/SCP_Enrollment_MA_2024_04.csv,1.010
35,ma_scp,ma-scp-2024-05.zip,2,1,1,SCP_Enrollment_MA_2024_05/SCP_Enrollment_MA_2024_05.csv,1.009


## Read Small Samples

Only the first few rows are read here. Full parsing belongs in the ingestion notebook/script, not in the inventory notebook.

In [6]:
def read_first_csv_from_zip(zip_path: Path, name_hint: str | None = None, nrows: int = 5) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as zf:
        csv_names = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if name_hint:
            hinted = [name for name in csv_names if name_hint.lower() in name.lower()]
            if hinted:
                csv_names = hinted
        if not csv_names:
            raise ValueError(f"No CSV found in {zip_path}")
        with zf.open(csv_names[0]) as handle:
            sample = pd.read_csv(handle, nrows=nrows)
    sample.attrs["source_zip"] = zip_path.name
    sample.attrs["source_csv"] = csv_names[0]
    return sample


ma_sample = read_first_csv_from_zip(sorted(MA_SCP_DIR.glob("*.zip"))[0], nrows=5)
print("MA SCP sample source:", ma_sample.attrs)
display(ma_sample)

cpsc_contract_sample = read_first_csv_from_zip(sorted(CPSC_DIR.glob("*.zip"))[0], name_hint="contract", nrows=5)
print("CPSC contract sample source:", cpsc_contract_sample.attrs)
display(cpsc_contract_sample)

cpsc_enrollment_sample = read_first_csv_from_zip(sorted(CPSC_DIR.glob("*.zip"))[0], name_hint="enrollment_info", nrows=5)
print("CPSC enrollment sample source:", cpsc_enrollment_sample.attrs)
display(cpsc_enrollment_sample)

MA SCP sample source:

 {'source_zip': 'ma-scp-2024-01.zip', 'source_csv': 'SCP_Enrollment_MA_2024_01/SCP_Enrollment_MA_2024_01.csv'}


,County,State,PLAN TYPE,SSA Code,FIPS Code,Enrolled
0,Autauga,AL,HCPP - 1833 Cost,1000,1001,.
1,Autauga,AL,HMO/HMOPOS,1000,1001,3689
2,Autauga,AL,LI NET Sponsor,1000,1001,.
3,Autauga,AL,Local PPO,1000,1001,3416
4,Autauga,AL,MSA,1000,1001,.


CPSC contract sample source: {'source_zip': 'monthly-enrollment-cpsc-2024-01.zip', 'source_csv': 'CPSC_Enrollment_2024_01/CPSC_Contract_Info_2024_01.csv'}


,Contract ID,Plan ID,Organization Type,Plan Type,Offers Part D,SNP Plan,EGHP,Organization Name,Organization Marketing Name,Plan Name,Parent Organization,Contract Effective Date
0,90091,NaN,HCPP - 1833 Cost,HCPP - 1833 Cost,No,No,No,UNITED MINE WORKERS OF AMERICA HLTH & RETIREMENT,United Mine Workers of America Health & Retirement,NaN,UMWA Health and Retirement Funds,02/01/1974
1,E3014,801.0,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,PSERS HOP PROGRAM,Pennsylvania Public School Employees Retirement Sy,PSERS Health Options Program (Employer PDP),Commonwealth of PA Pub Schools Retirement System,01/01/2007
2,E3014,802.0,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,PSERS HOP PROGRAM,Pennsylvania Public School Employees Retirement Sy,PSERS Health Options Program Value (Employer PDP),Commonwealth of PA Pub Schools Retirement System,01/01/2007
3,H0022,1.0,Demo,Medicare-Medicaid Plan HMO/HMOPOS,Yes,No,No,"BUCKEYE COMMUNITY HEALTH PLAN, INC.",Buckeye Health Plan - MyCare Ohio,Buckeye Health Plan - MyCare Ohio (Medicare-Medicaid Plan),Centene Corporation,05/01/2014
4,H0028,7.0,Local CCP,HMO/HMOPOS,Yes,Yes,No,"CHA HMO, INC.",Humana,Humana Gold Plus SNP-DE H0028-007 (HMO D-SNP),Humana Inc.,01/01/2013


CPSC enrollment sample source: {'source_zip': 'monthly-enrollment-cpsc-2024-01.zip', 'source_csv': 'CPSC_Enrollment_2024_01/CPSC_Enrollment_Info_2024_01.csv'}


,Contract Number,Plan ID,SSA State County Code,FIPS State County Code,State,County,Enrollment
0,E3014,801,58140,NaN,NaN,NaN,*
1,E3014,801,58300,NaN,NaN,NaN,*
2,E3014,801,58350,NaN,NaN,NaN,*
3,E3014,801,2275,NaN,NaN,NaN,*
4,E3014,801,2198,NaN,NaN,NaN,*


## Reference And Manual Export Sources

The Data.CMS.gov optional sources are registered for later targeted export/API work. They are not blockers for the first forecasting pipeline.

In [7]:
manual_links = MANUAL_DIR / "optional_source_links.csv"
if manual_links.exists():
    optional_sources = pd.read_csv(manual_links)
    display(optional_sources)
else:
    print("No optional source registry found yet.")

reference_files = inventory.loc[inventory["folder"].isin(["reference_docs", "manual_exports"])]
display(reference_files[["relative_path", "size_mb"]].sort_values("relative_path"))

,source,title,url,notes
0,data_cms,Medicare Geographic Variation - by National State and County,https://data.cms.gov/summary-statistics-on-use-and-payments/medicare-geographic-comparisons/medicare-geographic-vari...,Optional manual/API export for geographic spending and utilization context
1,data_cms,Medicare Physician and Other Practitioners - by Provider and Service,https://data.cms.gov/provider-summary-by-type-of-service/medicare-physician-other-practitioners/medicare-physician-o...,Optional large public-use file for specialty/procedure/payment proxy features
2,oig,HHS OIG Medicare Advantage prior authorization denial report,https://oig.hhs.gov/reports/all/2022/some-medicare-advantage-organization-denials-of-prior-authorization-requests-ra...,Optional benchmark/context source


,relative_path,size_mb
60,data/raw/manual_exports/medicare-geographic-variation-data-dictionary.pdf,0.182
61,data/raw/manual_exports/medicare-physician-provider-service-data-dictionary.pdf,0.106
62,data/raw/manual_exports/optional_source_links.csv,0.001
64,data/raw/reference_docs/cms-0057-f.pdf,3.371
65,data/raw/reference_docs/ma_step_therapy_hpms_memo_8_7_2018.pdf,0.055
66,data/raw/reference_docs/prior-authorization-metrics-reporting-overview-template.pdf,0.961


## Final Quality Gate

If these checks pass, the project is ready for notebook 02: MA enrollment EDA and normalized processed data creation.

In [8]:
quality_gate = {
    "raw_file_count": int(len(inventory)),
    "raw_size_gb": round(float(inventory["size_mb"].sum() / 1024), 3),
    "zip_count": int(len(zip_quality)),
    "bad_zip_count": int((~zip_quality["is_valid_zip"]).sum()),
    "ma_scp_months": int(coverage.loc[coverage["dataset"] == "ma_scp", "present"].sum()),
    "cpsc_months": int(coverage.loc[coverage["dataset"] == "cpsc", "present"].sum()),
    "expected_months": int(len(EXPECTED_MONTHS)),
}

quality_gate_df = pd.DataFrame([quality_gate])
quality_gate_df.to_csv(REPORT_TABLE_DIR / "data_inventory_quality_gate.csv", index=False)
display(quality_gate_df)

assert quality_gate["bad_zip_count"] == 0
assert quality_gate["ma_scp_months"] == quality_gate["expected_months"]
assert quality_gate["cpsc_months"] == quality_gate["expected_months"]

print("PASS: Raw data inventory and quality checks are ready for ingestion/EDA.")

,raw_file_count,raw_size_gb,zip_count,bad_zip_count,ma_scp_months,cpsc_months,expected_months
0,67,1.113,61,0,29,29,29


PASS: Raw data inventory and quality checks are ready for ingestion/EDA.
